# 04 · Retrieval evaluation and ablation (1 GPU)
Every variant on the same test set, the same frozen teacher index (except `base_student_own_index`, which uses its own
index of the same corpus) and the same top-100 retrieval. No training here.

Metrics: Recall@{1,5,10,20,50,100}, MRR@10, nDCG@10. Precision@K isn't reported: with ~1 positive per query,
P@100 ≤ 0.01, so recall@100 carries the same ranking information.

Input: `01` (data + teacher vectors), `02` + `03` (checkpoints), `03`'s `metrics.json` (all rows so far).

In [ ]:
# setup: clone/pull the repo on Kaggle, put it on sys.path, install what's missing (re-run to pull latest code)
import os, sys, importlib.util
REPO_URL = "https://github.com/pyther-hub/hlm-poc.git"
ON_KAGGLE = os.path.exists("/kaggle/working")
REPO = "/kaggle/working/hlm-poc" if ON_KAGGLE else os.path.abspath("..")
if ON_KAGGLE:
    if not os.path.exists(REPO):
        !git clone -q {REPO_URL} {REPO}
    else:
        !git -C {REPO} pull -q
sys.path.insert(0, REPO)

# HF token (higher rate limits, faster downloads). Kaggle: Add-ons -> Secrets -> add "HF_TOKEN", attach to notebook.
# Spawned workers inherit os.environ. Locally: export HF_TOKEN or `huggingface-cli login`.
if ON_KAGGLE and "HF_TOKEN" not in os.environ:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print(f"HF_TOKEN secret not found ({type(e).__name__}): downloads stay unauthenticated")

import transformers
from packaging.version import Version
if Version(transformers.__version__) < Version("4.56"):  # Qwen3 + the dtype= kwarg
    !pip install -q -U "transformers>=4.56"
    print("transformers upgraded: restart the kernel, then re-run this cell")
for pkg in ("datasets", "pyarrow", "accelerate"):
    if importlib.util.find_spec(pkg) is None:
        !pip install -q {pkg}

%load_ext autoreload
%autoreload 2

## Control panel

In [ ]:
import os
import shutil
import torch
from model import get_device
from teacher import TEACHER_DEFAULTS as T

DEBUG = True    # use the same value as in 00..03

# --- paths. On Kaggle, earlier notebooks' outputs are attached as inputs (read only); set the slugs to yours.
#     Off Kaggle everything lives under the repo, so every IN_* is the same dir.
ROOT = "/kaggle/working" if ON_KAGGLE else REPO
MODE = "debug" if DEBUG else "full"
KAGGLE_IN = "/kaggle/input/notebooks/tensorpanda231"
IN_01 = f"{KAGGLE_IN}/hlm-01-teacher-embeddings" if ON_KAGGLE else REPO
IN_02 = f"{KAGGLE_IN}/hlm-02-align-prune" if ON_KAGGLE else REPO
IN_03 = f"{KAGGLE_IN}/hlm-03-refine" if ON_KAGGLE else REPO
DATA_PATH = f"{IN_01}/cache/{MODE}/data.json"
EMB_DIR = f"{IN_01}/cache/{MODE}/emb/teacher"
RUN_DIR = f"{ROOT}/runs/{MODE}"
PREV_METRICS = f"{IN_03}/runs/{MODE}/metrics.json"

# --- every variant: name -> checkpoint dir
STUDENTS = ("aligned", "prog_l14", "prog_l7")
MODELS = {**{n: f"{IN_02}/runs/{MODE}/{n}/final" for n in STUDENTS},
          **{f"{n}_cr": f"{IN_03}/runs/{MODE}/{n}_cr/final" for n in STUDENTS}}

# --- latency: name -> (model, max_len, prefix); the first is the speedup reference
LATENCY_MODELS = {"teacher_4b": (T["model"], T["query_max_len"], T["query_prompt"]),
                  **{n: (path, 32, "") for n, path in MODELS.items()}}
LATENCY = dict(n_queries=2000, batch_sizes=(1, 8, 32, 128))

# --- evaluation (evaluation.evaluate_model kwargs): student queries vs the frozen teacher corpus, top 100
EVAL = dict(splits=("val", "test"), dim=T["dim"], max_query_len=32, query_prefix="", device=get_device(), dtype="fp16")

if DEBUG:
    LATENCY.update(n_queries=200)

## 1 · Inputs

In [ ]:
from data import load_json
from evaluation import load_index

# rows accumulate across notebooks: start from the previous notebook's metrics.json (copied once)
os.makedirs(RUN_DIR, exist_ok=True)
if not os.path.exists(f"{RUN_DIR}/metrics.json") and os.path.exists(PREV_METRICS):
    shutil.copy(PREV_METRICS, f"{RUN_DIR}/metrics.json")
data = load_json(DATA_PATH)
index = load_index(data, EMB_DIR, ("val", "test"))
missing = [n for n, p in MODELS.items() if not os.path.exists(p)]
assert not missing, f"checkpoints missing: {missing}"

## 2 · Retrieval quality, all variants
Rows already in `metrics.json` are reused; any missing one is evaluated now. retention = nDCG@10 / teacher nDCG@10.

In [ ]:
from evaluation import summarize
from run_exp import evaluate_runs

metrics = evaluate_runs(MODELS, data, index, RUN_DIR, EVAL)
for split in ("val", "test"):
    summarize(RUN_DIR, split)
    print()

## 3 · Efficiency: parameters, memory, latency, throughput
Query encoding only (documents are pre-embedded). fp16 on one GPU; memory = parameter bytes in fp16.
Also mean cos(q, d+) on val: how close each model's query vectors sit to the teacher's positive passage.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display
from run_exp import exp_latency

res = exp_latency(LATENCY_MODELS, data, index, RUN_DIR, EVAL["device"], T["dim"], **LATENCY)
print(f"\n{'model':<16}{'params':>9}{'fp16 MB':>9}")
for name, lat in res["latency"].items():
    print(f"{name:<16}{lat['params'] / 1e6:>8.0f}M{lat['params'] * 2 / 1e6:>9.0f}")
for fig in res["figs"]:
    display(fig)
plt.close("all")

## 4 · Ablations (test, from the rows above)
- **alignment**: untrained student (own index) → aligned
- **pruning**: quality lost per cut
- **refinement**: quality recovered by contrastive refinement at each depth

In [ ]:
from run_exp import plot_bars, save_fig

STEPS = [("alignment", "base_student_own_index", "aligned"),
         ("prune 28->14", "aligned", "prog_l14"),
         ("prune 14->7", "prog_l14", "prog_l7"),
         *[(f"refine {n}", n, f"{n}_cr") for n in STUDENTS]]
print(f"{'step':<16}{'from':<24}{'to':<14}" + "".join(f"{k:>16}" for k in ("ndcg@10", "recall@100")))
for step, a, b in STEPS:
    if a in metrics and b in metrics:
        d = {k: (metrics[a]["test"][k], metrics[b]["test"][k]) for k in ("ndcg@10", "recall@100")}
        print(f"{step:<16}{a:<24}{b:<14}" + "".join(f"{x:.3f}->{y:.3f}".rjust(16) for x, y in d.values()))

names = ["teacher", *(n for n in ("base_student_own_index", *MODELS) if n in metrics)]
display(save_fig(plot_bars(metrics, names, title="all variants (test)"), RUN_DIR, "all_variants"))
plt.close("all")